# Beating Your Own Best Agent Is The Wrong Test

Yesterday I found a candidate agent that beat my tracked submission **93–100% across three
independent seed sets**, 78 wins in 80 games. Obvious upgrade. I nearly submitted it.

Then I ran it against a *panel* of five agents instead of just against my own bot, and the
ranking inverted:

| agent | win rate vs the whole panel |
|---|---|
| my incumbent | **70.0%** |
| **the "upgrade"** | **40.0%** |

The candidate beat my bot almost every single game and was still much worse against the
field. This notebook is about why that happens, why it is not a fluke of this competition,
and the cheap fix.

*(High-school student; this is an economics/OR project for me. Corrections welcome.)*

## The mechanism: this field is non-transitive

Rock-paper-scissors is the toy version. Kaggriculture has the real version, and it is easy
to produce: an agent's result is decided largely by **which products it floods and when**.
Two agents that dump different goods on different days will beat and lose to *different*
third parties, regardless of which of them beats the other.

Here is the actual round-robin, agents anonymised (see the note at the bottom):

| | vs A | vs B | vs C | vs D | vs E | **overall** |
|---|---|---|---|---|---|---|
| **A** | — | 100% | 100% | 100% | 12.5% | **82.5%** |
| **B** | 0% | — | 100% | 100% | 100% | 70.0% |
| **C** | 0% | 0% | — | 100% | 100% | 50.0% |
| **E** | 87.5% | 100% | 12.5% | 0% | — | 40.0% |
| **D** | 0% | 0% | 0% | — | 100% | 30.0% |

Read the cycle: **E beats A, A beats B, B beats E.** There is no total order here. "Better"
is not a property an agent has on its own — it is a property of an agent *against a
distribution of opponents*.

My incumbent was B. My "upgrade" was E. E beats B 100%. E is two places below B overall.

## Why the usual workflow walks straight into it

The default loop is: keep a champion, try a challenger, promote the challenger if it beats
the champion. That loop is correct when strength is a scalar. Here it is a random walk —
you can promote your way around a cycle and arrive back where you started, having "improved"
at every step.

Worse, it is *self-reinforcing*: each promotion makes your champion better tuned against the
previous champion specifically, which is the one opponent guaranteed never to appear on the
ladder again.

## The fix, in two parts

**1. Rank against a panel, never against your incumbent.** Assemble 4–6 opponents that are
genuinely *different* from each other, and score every candidate against all of them. Cost
is linear in panel size and it is the cheapest insurance in the competition.

Note "genuinely different" is doing real work. A lot of the strong public agents are
byte-identical copies of a few underlying action tapes — in my corpus one tape had **seven**
public copies. A panel of seven clones is a panel of one. Dedup by hashing the decoded
action data, not the file text.

**2. With two tracked submission slots, think in portfolios, not in winners.**

An important correction here, owed to **Sanskar Omar** in the comments: ratings are
per-submission and fully independent. Pairing A and E does **not** shield either one — your
A still meets other people's E-types on the ladder and loses those games, and no bot you
own ever protects another. If "covering pair" read as "the two agents help each other", that
was my sloppiness; they never interact.

The defensible version is a portfolio argument. Kaggle ranks you by your **best**
submission, so the quantity to maximise with two slots is `E[max(rating_A, rating_B)]`
under uncertainty about the field's composition. Each bot fights alone and finds its own
level; you win if *at least one* of them happens to be well-matched to the meta that
actually exists. Two agents with the same blind spot fail together; two with different
blind spots fail on different futures. That is a hedge, and it changes nothing about any
individual game.

The honest boundary of the argument: if the field mix is *balanced* — plenty of A-eaters
and E-eaters both — then both slots sit mid-table and the hedge buys nothing over simply
fielding the single best-EV agent twice as it improves. Diversification pays only when the
meta is concentrated or uncertain. My read is that it currently is (a surprising fraction
of strong public agents are byte-identical copies of a few underlying tapes), but that is
a judgment about today's field, not a theorem.

In [ ]:
# A minimal panel scorer. Point AGENTS at your own modules and run.
# Every matchup plays BOTH seatings on a fixed seed set -- seat order matters in this game
# because market orders resolve in player order, so a one-sided test is a biased one.
import itertools, statistics

def play(make_a, make_b, seed, swap):
    from kaggle_environments import make
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": seed})
    env.run([make_b(), make_a()] if swap else [make_a(), make_b()])
    f = env.steps[-1]
    i = 1 if swap else 0
    a, b = float(f[i].reward or 0), float(f[1-i].reward or 0)
    return 1.0 if a > b else (0.5 if a == b else 0.0)

def head_to_head(make_a, make_b, seeds):
    return statistics.mean(play(make_a, make_b, s, sw) for s in seeds for sw in (False, True))

def panel_table(agents, seeds):
    """agents: {name: zero-arg factory returning an agent callable}"""
    names = list(agents)
    grid = {a: {} for a in names}
    for a, b in itertools.permutations(names, 2):
        if b in grid[a]:
            continue
        w = head_to_head(agents[a], agents[b], seeds)
        grid[a][b], grid[b][a] = w, 1.0 - w
    print(f"{'agent':<16}" + "".join(f"{n[:9]:>11}" for n in names) + f"{'OVERALL':>11}")
    for a in sorted(names, key=lambda x: -statistics.mean(grid[x].values())):
        row = "".join(f"{grid[a].get(b, float('nan')):>10.0%} " if b != a else f"{'--':>11}"
                      for b in names)
        print(f"{a:<16}{row}{statistics.mean(grid[a].values()):>10.0%}")
    return grid

def best_covering_pair(grid, threshold=0.5):
    """The pair whose UNION beats the most panel members. NOTE (see the correction above):
    this is a PORTFOLIO hedge on E[max(rating_A, rating_B)] -- the two bots never help each
    other in any game. Coverage is a tiebreaker among individually strong pairs, not the
    selection criterion on its own."""
    names = list(grid)
    best = None
    for a, b in itertools.combinations(names, 2):
        covered = sum(1 for t in names if t not in (a, b)
                      and max(grid[a].get(t, 0), grid[b].get(t, 0)) > threshold)
        if best is None or covered > best[0]:
            best = (covered, a, b)
    print(f"\nbest covering pair: {best[1]} + {best[2]}  "
          f"(covers {best[0]} of {len(names)-2} other panel members)")
    return best

print("Ready. Build an AGENTS dict of factories and call panel_table(AGENTS, range(1000,1008)).")

## Two honest caveats

**The panel is not the field, and an unweighted panel is a specific bet.** It says every
opponent is equally likely. On a real ladder they are not — if one tape has seven public
copies and another has one, you will meet the first far more often. I re-ranked my own
candidates weighted by public clone counts and **the order changed again**. I could not
resolve which ranking is right, because clone counts are a proxy for ladder composition, not
a measurement of it. When the weighted and unweighted rankings disagree, that is an argument
for a covering pair rather than for trusting either number.

**Sample size.** Every cell above is n=16 (8 seeds x 2 seatings). That is enough to separate
100% from 0%, and nowhere near enough to separate 55% from 45%. Do not promote on a 10-point
difference at this n.

## On the anonymisation

A–E are real public agents from this competition and the numbers are from real runs, but I
have not named them while the competition is live. Publishing the method helps everyone
evaluate better; publishing a ranked shopping list of which public tape is strongest just
does other people's agent-selection for them. I would rather contribute the first thing.

Happy to be told the anonymisation is excessive — if the consensus is that naming them is
more useful than not, I will edit it in.